# Stage 5 VLM Benefit Report

Сборка финального отчёта: no-VLM vs direct VLM vs hybrid. Скопируйте `drop_in/scripts/build_vlm_benefit_report.py` в `scripts/` репозитория.

In [ ]:

from pathlib import Path
import subprocess, json, os, shutil, csv
import pandas as pd
from IPython.display import display, Markdown

REPO_URL = os.environ.get("REPO_URL", "https://github.com/konstRyaz/vlm-for-insulator-defect-detection.git")
REPO_DIR = Path(os.environ.get("REPO_DIR", "/kaggle/working/vlm-for-insulator-defect-detection"))
DATASET_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/kostyaryazanov/idid-coco-v3"),
    Path("/kaggle/input/idid-coco-v3"),
    Path("/kaggle/input"),
]
TRAIN_REL = Path("stage3_regrouped_v2/train_balanced/vlm_labels_v1_train_balanced_v2.annotated.jsonl")
VAL_REL = Path("stage3_regrouped_v2/val/vlm_labels_v1_val_v2.annotated.jsonl")

def sh(args, cwd=None, check=True):
    print("$", " ".join(str(a) for a in args))
    p = subprocess.run([str(a) for a in args], cwd=str(cwd) if cwd else None, text=True, capture_output=True)
    if p.stdout:
        print(p.stdout)
    if p.stderr:
        print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Command failed {p.returncode}: {' '.join(str(a) for a in args)}")
    return p

def find_data_root():
    for root in DATASET_ROOT_CANDIDATES:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    for root in Path('/kaggle/input').glob('**') if Path('/kaggle/input').exists() else []:
        if (root/TRAIN_REL).exists() and (root/VAL_REL).exists():
            return root
    raise FileNotFoundError('Could not find stage3_regrouped_v2 train/val JSONL')


In [ ]:
if not REPO_DIR.exists():
    sh(['git','clone',REPO_URL,str(REPO_DIR)])
sh(['python','-m','pip','install','-q','pandas'], cwd=REPO_DIR)


In [ ]:
sh([
    'python','scripts/build_vlm_benefit_report.py',
    '--non-vlm-leaderboard', 'reports/next_research/non_vlm_baselines/leaderboard_non_vlm.csv',
    '--current-registry', 'reports/operation_next/experiment_registry.csv',
    '--structured-eval-root', 'reports/next_research/structured_output_eval',
    '--out-dir', 'reports/next_research/vlm_benefit',
], cwd=REPO_DIR)


In [ ]:
report = REPO_DIR/'reports/next_research/vlm_benefit/vlm_vs_non_vlm_benefit_report.md'
print(report.read_text(encoding='utf-8'))
benefit = pd.read_csv(REPO_DIR/'reports/next_research/vlm_benefit/benefit_matrix.csv')
display(benefit)


In [ ]:
archive = Path('/kaggle/working/stage5_vlm_benefit_report.tar.gz')
if archive.exists(): archive.unlink()
sh(['tar','-czf',archive,'-C',REPO_DIR,'reports/next_research/vlm_benefit'], check=True)
print('Archive:', archive)
